In [16]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = Path("/Users/samermarroki/Downloads/Dataset 1 (Bank Marketing)/bank_marketing.csv")
README_PATH = Path("/Users/samermarroki/Downloads/Dataset 1 (Bank Marketing)/README.txt")
OUTPUT_DIR = Path("/Users/samermarroki/Documents/Playground/bank_marketing_outputs")


def describe_dataset(df: pd.DataFrame) -> None:
    print("\n=== DATASET CHARACTERISTICS ===")
    print(f"Dataset file: {DATA_PATH}")
    print(f"README file:   {README_PATH}")
    print("File type: CSV")
    print("Import method: pandas.read_csv(..., sep=';')")
    print(f"Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")

    print("\nColumn data types:")
    print(df.dtypes.to_string())

    print("\nMissing values before cleaning:")
    print(df.isna().sum().to_string())

    print("\n'unknown' placeholder counts:")
    for col in df.select_dtypes(include=["object", "string"]).columns:
        count = df[col].astype("string").str.strip().str.lower().eq("unknown").sum()
        if count > 0:
            print(f"{col}: {count}")

    print(f"\nExact duplicate rows: {df.duplicated().sum()}")


def clean_and_wrangle(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.columns = (
        cleaned.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    )

    string_cols = cleaned.select_dtypes(include=["object", "string"]).columns
    for col in string_cols:
        cleaned[col] = cleaned[col].astype("string").str.strip().str.lower()
        cleaned[col] = cleaned[col].replace({"": pd.NA, "nan": pd.NA})

    unknown_cols = ["job", "education", "contact", "poutcome"]
    for col in unknown_cols:
        cleaned[f"{col}_was_unknown"] = (
            cleaned[col].eq("unknown").fillna(False).astype(int)
        )
        cleaned[col] = cleaned[col].replace("unknown", pd.NA)

    cleaned["default"] = cleaned["default"].replace({"unknown": pd.NA})

    cleaned = cleaned.drop_duplicates().reset_index(drop=True)

    cleaned["age"] = cleaned.groupby("job")["age"].transform(
        lambda s: s.fillna(s.median())
    )
    cleaned["age"] = cleaned["age"].fillna(cleaned["age"].median())

    for col in ["job", "education", "default"]:
        cleaned[col] = cleaned[col].fillna(cleaned[col].mode(dropna=True).iat[0])

    cleaned["contact"] = cleaned["contact"].fillna("missing")
    cleaned["poutcome"] = cleaned["poutcome"].fillna("missing")

    return cleaned


def transform_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    transformed = df.copy()

    transformed["was_previously_contacted"] = (transformed["pdays"] != -1).astype(int)
    transformed["effective_pdays"] = transformed["pdays"].replace(-1, np.nan)
    transformed["balance_per_contact"] = transformed["balance"] / transformed["campaign"].clip(lower=1)
    transformed["duration_per_contact"] = transformed["duration"] / transformed["campaign"].clip(lower=1)
    transformed["has_any_loan"] = (
        (transformed["housing"] == "yes") | (transformed["loan"] == "yes")
    ).astype(int)

    month_order = {
        "jan": 1,
        "feb": 2,
        "mar": 3,
        "apr": 4,
        "may": 5,
        "jun": 6,
        "jul": 7,
        "aug": 8,
        "sep": 9,
        "oct": 10,
        "nov": 11,
        "dec": 12,
    }
    transformed["month_num"] = transformed["month"].map(month_order).astype("Int64")
    transformed["quarter"] = pd.cut(
        transformed["month_num"],
        bins=[0, 3, 6, 9, 12],
        labels=["q1", "q2", "q3", "q4"],
    )

    transformed["age_group"] = pd.cut(
        transformed["age"],
        bins=[0, 25, 35, 50, 65, np.inf],
        labels=["18-25", "26-35", "36-50", "51-65", "65+"],
        include_lowest=True,
    )
    transformed["balance_band"] = pd.qcut(
        transformed["balance"].rank(method="first"),
        q=4,
        labels=["low", "medium", "high", "very_high"],
    )
    transformed["campaign_band"] = pd.cut(
        transformed["campaign"],
        bins=[0, 1, 3, 6, np.inf],
        labels=["1_contact", "2_to_3", "4_to_6", "7_plus"],
        include_lowest=True,
    )

    numeric_cols = [
        "age",
        "balance",
        "day",
        "duration",
        "campaign",
        "previous",
        "balance_per_contact",
        "duration_per_contact",
    ]
    for col in numeric_cols:
        col_min = transformed[col].min()
        col_max = transformed[col].max()
        if pd.notna(col_min) and pd.notna(col_max) and col_max != col_min:
            transformed[f"{col}_minmax"] = (transformed[col] - col_min) / (col_max - col_min)
        else:
            transformed[f"{col}_minmax"] = 0.0

    aggregation = (
        transformed.groupby("deposit", dropna=False)
        .agg(
            customers=("deposit", "size"),
            avg_age=("age", "mean"),
            avg_balance=("balance", "mean"),
            avg_duration=("duration", "mean"),
            avg_campaign_contacts=("campaign", "mean"),
            conversion_rate_previous_success=("poutcome", lambda s: (s == "success").mean()),
        )
        .round(2)
        .reset_index()
    )

    return transformed, aggregation


def reduce_redundancy(df: pd.DataFrame) -> pd.DataFrame:
    reduced = df.copy()
    removed_columns = []

    if "month_num" in reduced.columns and "month" in reduced.columns:
        reduced = reduced.drop(columns=["month"])
        removed_columns.append("month")

    if "effective_pdays" in reduced.columns and "pdays" in reduced.columns:
        reduced = reduced.drop(columns=["pdays"])
        removed_columns.append("pdays")

    print("\nColumns removed to reduce redundancy:")
    print(removed_columns if removed_columns else "None")

    return reduced


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(DATA_PATH, sep=";")
    describe_dataset(df)

    cleaned = clean_and_wrangle(df)
    print("\n=== AFTER CLEANING ===")
    print(cleaned.isna().sum().to_string())

    transformed, aggregation = transform_features(cleaned)
    reduced = reduce_redundancy(transformed)

    cleaned_path = OUTPUT_DIR / "bank_marketing_cleaned.csv"
    transformed_path = OUTPUT_DIR / "bank_marketing_transformed.csv"
    reduced_path = OUTPUT_DIR / "bank_marketing_reduced.csv"
    aggregation_path = OUTPUT_DIR / "bank_marketing_aggregation_by_deposit.csv"

    cleaned.to_csv(cleaned_path, index=False)
    transformed.to_csv(transformed_path, index=False)
    reduced.to_csv(reduced_path, index=False)
    aggregation.to_csv(aggregation_path, index=False)

    print("\n=== OUTPUT FILES ===")
    print(cleaned_path)
    print(transformed_path)
    print(reduced_path)
    print(aggregation_path)

    print("\n=== SAMPLE AGGREGATION ===")
    print(aggregation.to_string(index=False))


if __name__ == "__main__":
    main()





=== DATASET CHARACTERISTICS ===
Dataset file: /Users/samermarroki/Downloads/Dataset 1 (Bank Marketing)/bank_marketing.csv
README file:   /Users/samermarroki/Downloads/Dataset 1 (Bank Marketing)/README.txt
File type: CSV
Import method: pandas.read_csv(..., sep=';')
Dimensions: 45211 rows x 17 columns

Column data types:
age          float64
job              str
marital          str
education        str
default          str
balance        int64
housing          str
loan             str
contact          str
day            int64
month            str
duration       int64
campaign       int64
pdays          int64
previous       int64
poutcome         str
deposit          str

Missing values before cleaning:
age          1339
job             0
marital         0
education       0
default      1306
balance         0
housing         0
loan            0
contact      1383
day             0
month           0
duration        0
campaign        0
pdays           0
previous        0
poutcome        0
